In [104]:
import pandas as pd



In [105]:
df=pd.read_csv("IMDB Dataset.csv")

In [106]:
df.shape

(50000, 2)

In [107]:
df.head()




,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [108]:
df.drop_duplicates(inplace=True)

In [109]:
df.shape

(49582, 2)

In [110]:
# Converting data into lowercase

df["review"]=df["review"].str.lower()

In [111]:
# Removing the url

import re

def remove_url(text):
  text=re.sub(r"http\S+","",text)
  return text
df["review"]=df["review"].apply(remove_url)

In [112]:
# Remove Punctuation

def remove_punctuate(text):
  text=re.sub(r"[^A-za-z0-9\s]","",text)
  return text
df["review"]=df["review"].apply(remove_punctuate)

In [113]:
# HTML Tags

def remove_html(text):
  text=re.sub(r"<.*?>","",text)
  return text
df["review"]=df["review"].apply(remove_html)

In [114]:
import nltk

from nltk.corpus import stopwords
nltk.download('stopwords')

from nltk.tokenize import word_tokenize

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [115]:
# remove stopwords

def remove_stopwords(text):
  tokens=word_tokenize(text)
  stop_words=stopwords.words("english")

  for word in stop_words:
    if word in stop_words:
      text=text.replace(word,"")
  return text

  df["review"]=df["review"].apply(remove_stopwords)

In [116]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


In [117]:
# Stemming

def stemming(text):
  ps=PorterStemmer()
  stemmed_words=[]

  tokens=word_tokenize(text)
  for token in tokens:
    stemmed_token=ps.atem(token)
    stemmed_words.append(stemmed_token)

  return "".join(stemmed_words)
  df["review"]=df["review"].apply(stemming)

In [118]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


In [119]:
df.shape

(49582, 2)

In [120]:
# Encodding

from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df["sentiment"]=le.fit_transform(df["sentiment"])

In [121]:
y=df["sentiment"]
y.head()

,sentiment
0,1
1,1
2,1
3,0
4,1


In [122]:
# Vectorization


from sklearn.feature_extraction.text import TfidfVectorizer
tf=TfidfVectorizer(max_features=5000)
X=tf.fit_transform(df["review"])
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5696465 stored elements and shape (49582, 5000)>

In [123]:


from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [124]:
X_train.shape

(39665, 5000)

In [125]:
X_test.shape

(9917, 5000)

In [128]:
# Dataset & Dataloader

import torch
from torch.utils.data import TensorDataset,DataLoader

X_train=X_train.toarray()
X_test=X_test.toarray()

In [129]:
train_set=TensorDataset(torch.from_numpy(X_train).float(),
                       torch.from_numpy(y_train.to_numpy()).float()
                       )

test_set=TensorDataset(torch.from_numpy(X_test).float(),
                       torch.from_numpy(y_test.to_numpy()).float()
                       )

In [131]:
train_loader=DataLoader(train_set,shuffle=True,batch_size=64)
test_loader=DataLoader(test_set,shuffle=True,batch_size=64)

In [132]:
import torch.nn as nn
import torch.optim as optim


In [ ]:
class RNN(nn.Module):
  def __init__(self,input_size,hidden_size=128,num_layers=1):

    super().__init__() # Added parentheses

    self.hidden_size=hidden_size
    self.num_layers=num_layers

    # RNN layer
    self.rnn=nn.RNN(input_size,hidden_size,num_layers,batch_first=True)

    # Fully connected layer
    self.fc=nn.Linear(hidden_size,1) # Corrected to nn.Linear

  def forward(self,x):
   

    h0=torch.zeros(self.num_layers,x.size(0),self.hidden_size).to(x.device)

    # Pass input and initial hidden state to RNN
    out, _ = self.rnn(x, h0) # 

    out = self.fc(out[:, -1, :]) # 
    return out

In [135]:
input_size=X_train.shape[1]
model=RNN(input_size)
criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters())

In [137]:
# Trainning the RNN

epochs=10
for epoch in range(epochs):
  model.train()

  for xb,yb in train_loader:
    optimizer.zero_grad()

    xb=xb.unsqueeze(1) # Add singleton direction

    outputs=model(xb)
    outputs=torch.sigmoid(outputs).squeeze() # Corrected: 'sequeeze' to 'squeeze' and applied as a method

    loss=criterion(outputs,yb)
    loss.backward()
    optimizer.step()

print(f"epoch={epoch+1}/{epochs} & loss={loss.item()}")

epoch=10/10 & loss=0.1110198125243187


In [141]:
# Evaluate
model.eval()
with torch.no_grad():
  correct_vals=0
  tot_vals=0

  for xb,yb in test_loader:
    xb=xb.unsqueeze(1)

    # Correctly calculate outputs for the current test batch
    outputs=model(xb)
    predicted=(torch.sigmoid(outputs).squeeze()>0.5).float()

    tot_vals+=yb.size(0)
    correct_vals+=(predicted==yb).sum().item()

  # Corrected accuracy calculation
  print(f"accuracy={(correct_vals/tot_vals)*100}")

accuracy=87.57688817182616
